In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(viridis)
    library(glmmTMB)
    library(ggeffects)
    library(data.table)
    library(patchwork)
    library(purrr)
})

# Load data

In [ ]:
df_all = readRDS("../data/human_fetal_atlas_df_glmm_embryo.RDS")
dim(df_all)

In [ ]:
df_all = df_all %>%
            filter(normalized_nuc_score_0.7_0.4 > 0 & !is.na(Organ_cell_lineage))
dim(df_all)

In [ ]:
df_cell_blood = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas/df_cell_blood.RDS") %>% filter(sample %in% df_all$sample)
df_cell_endothelial = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas/df_cell_endothelial.RDS") %>% filter(sample %in% df_all$sample)
df_cell_epithelial = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas/df_cell_epithelial.RDS") %>% filter(sample %in% df_all$sample)

In [ ]:
df_cell_blood_c = df_cell_blood %>%
    select(sample, Organ, Blood_cell_name) %>%
    rename(Organ_cell_lineage = Blood_cell_name)

df_cell_endothelial_c = df_cell_endothelial %>%
    select(sample, Organ, Endothelial_cell_name) %>%
    rename(Organ_cell_lineage = Endothelial_cell_name)

df_cell_epithelial_c = df_cell_epithelial %>%
    select(sample, Organ, Epithelial_cell_name) %>%
    rename(Organ_cell_lineage = Epithelial_cell_name)

df_cell_comb = rbind(df_cell_blood_c, df_cell_endothelial_c, df_cell_epithelial_c)

In [ ]:
df_all$celltype_fine = df_all$celltype
df_all[df_all$sample %in% df_cell_comb$sample, "celltype_fine"] = df_cell_comb$Organ_cell_lineage

In [ ]:
length(unique(df_all$celltype_fine))

# Analysis

In [ ]:
df_ctype = df_all %>%
    group_by(Organ, Fetus_id, Development_day, celltype) %>%
    summarise(med_score_0.7_0.4 = median(score_pred_glmm_0.7_0.4), 
              n = n()) %>%
    filter(n >= 50) %>%
    na.omit() %>%
    as.data.frame()

In [ ]:
df_ctype = df_ctype %>%
    mutate(organ_ctype = paste0(Organ, '-', celltype))

In [ ]:
## Figure 7A
options(repr.plot.width=5, repr.plot.height=6, repr.plot.res=300)
set.seed(11)
df_ctype %>%
    ggplot(., aes(x = reorder(Organ, med_score_0.7_0.4, FUN = median, decreasing = T), 
                  y = med_score_0.7_0.4)) + 
        geom_boxplot(aes(fill = Organ), outlier.shape = NA, lwd = 0.75, fatten = 0.75, alpha = 0.9) + 
        labs(x = "", y = "Nucleolar fraction score") + 
        monocle3:::monocle_theme_opts() + 
        theme(legend.position = "none") + 
        theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
            axis.text = element_text(size = 15, color = "black"),
            axis.title = element_text(size = 16),
            axis.line = element_line(linewidth = 1.5, color = "black"),
            legend.position = "none",
            legend.title = element_text(face = "bold")
        )

## Blood lineage

In [ ]:
df_sub = df_all %>%
    filter(sample %in% df_cell_blood$sample) %>%
    mutate(Blood_cell_name = df_cell_blood$Blood_cell_name,
           Blood_trajectory_umap_1 = df_cell_blood$Blood_trajectory_umap_1,
           Blood_trajectory_umap_2 = df_cell_blood$Blood_trajectory_umap_2) 

In [ ]:
min_lim = quantile(df_sub$score_pred_glmm_0.7_0.4, 0.05)
mid_lim = quantile(df_sub$score_pred_glmm_0.7_0.4, 0.5)
max_lim =quantile(df_sub$score_pred_glmm_0.7_0.4, 0.95)

options(repr.plot.width=6.5, repr.plot.height=5, repr.plot.res=300)
df_sub %>%
    ggplot(., aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2)) + 
        geom_point(aes(color = Blood_cell_name), size = 0.2) + 
        monocle3:::monocle_theme_opts() + guides(colour = guide_legend(override.aes = list(size=3)))

options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)
df_sub %>%
    ggplot(., aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2)) + 
        geom_point(aes(color = score_pred_glmm_0.7_0.4), size = I(0.3), stroke= I(0.3 / 2)) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = mid_lim,
                               limits = c(min_lim, max_lim), oob=scales::squish, name = "Score") + 
        monocle3:::monocle_theme_opts() +
        theme(aspect.ratio = 1)

In [ ]:
## Figure 7B

options(repr.plot.width=2.7, repr.plot.height=4, repr.plot.res=300)

df_sub_sub = df_sub %>%
    filter(Blood_cell_name == "Erythroblasts") 
celltype_plot = "Pancreas"

df_sub_sub %>%
    ggplot() + 
        geom_point(data = df_sub_sub %>% filter(Organ != celltype_plot), 
                   aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2),
                   color = "grey80", size = I(0.2), stroke= I(0.2 / 2)) + 
        geom_point(data = df_sub_sub %>% filter(Organ == celltype_plot),
                    aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2, 
                        color = score_pred_glmm_0.7_0.4), 
                     size = I(0.45), stroke= I(0.45 / 2)) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = mid_lim,
                               limits = c(min_lim, max_lim), oob=scales::squish) + 
        xlim(c(NA, 0)) + ylim(c(NA, 5.2)) + 
        labs(color = "Nucleolar\nfraction score") + 
        monocle3:::monocle_theme_opts() + 
        theme(legend.position = "none")

In [ ]:
## Figure 7B

options(repr.plot.width=2.7, repr.plot.height=4, repr.plot.res=300)

df_sub_sub = df_sub %>%
    filter(Blood_cell_name == "Erythroblasts") 
celltype_plot = "Adrenal"

df_sub_sub %>%
    ggplot() + 
        geom_point(data = df_sub_sub %>% filter(Organ != celltype_plot), 
                   aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2),
                   color = "grey80", size = I(0.2), stroke= I(0.2 / 2)) + 
        geom_point(data = df_sub_sub %>% filter(Organ == celltype_plot),
                    aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2, 
                        color = score_pred_glmm_0.7_0.4), 
                     size = I(0.45), stroke= I(0.45 / 2)) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = mid_lim,
                               limits = c(min_lim, max_lim), oob=scales::squish) + 
        xlim(c(NA, 0)) + ylim(c(NA, 5.2)) + 
        labs(color = "Nucleolar\nfraction score") + 
        monocle3:::monocle_theme_opts() + 
        theme(legend.position = "none")

In [ ]:
## Figure 7B

options(repr.plot.width=2.7, repr.plot.height=4, repr.plot.res=300)

df_sub_sub = df_sub %>%
    filter(Blood_cell_name == "Erythroblasts") 
celltype_plot = "Liver"

df_sub_sub %>%
    ggplot() + 
        geom_point(data = df_sub_sub %>% filter(Organ != celltype_plot), 
                   aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2),
                   color = "grey80", size = I(0.2), stroke= I(0.2 / 2)) + 
        geom_point(data = df_sub_sub %>% filter(Organ == celltype_plot),
                    aes(x = Blood_trajectory_umap_1, y = Blood_trajectory_umap_2, 
                        color = score_pred_glmm_0.7_0.4), 
                     size = I(0.3), stroke= I(0.3 / 2)) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = mid_lim,
                               limits = c(min_lim, max_lim), oob=scales::squish) + 
        xlim(c(NA, 0)) + ylim(c(NA, 5.2)) + 
        labs(color = "Nucleolar\nfraction score") + 
        monocle3:::monocle_theme_opts() + 
        theme(legend.position = "none")

In [ ]:
## Figure 7B

set.seed(163) 
random_turbo <- sample(viridis::turbo(14))

ctype = "Erythroblasts"

options(repr.plot.width=2.5, repr.plot.height=3, repr.plot.res=300)
df_sub = df_all %>%
    filter(grepl(ctype, Organ_cell_lineage))

organ_ordered = df_sub %>%
                group_by(Organ) %>%
                summarise(med_score_pred_glmm_0.7_0.4 = median(score_pred_glmm_0.7_0.4),
                          n = n()) %>%
                arrange(med_score_pred_glmm_0.7_0.4) %>%
                pull(Organ)

ylim1 = boxplot.stats(df_sub %>%
                        filter(Organ == organ_ordered[1]) %>%
                        pull(score_pred_glmm_0.7_0.4))$stats[c(1, 5)] * 1.6
ylim1[1] = boxplot.stats(df_sub %>%
                        filter(Organ == organ_ordered[length(organ_ordered)]) %>%
                        pull(score_pred_glmm_0.7_0.4))$stats[1] * 0.85

df_sub %>%
    mutate(Organ = factor(Organ, levels = organ_ordered)) %>%
    filter(Organ %in% c("Adrenal", "Liver", "Pancreas")) %>%
    ggplot(., aes(x = Organ, y = score_pred_glmm_0.7_0.4)) + 
        geom_violin(aes(fill = Organ), alpha = 0.9, adjust = 2, width = 0.9) + 
        geom_boxplot(width = 0.3, outlier.shape = NA, fatten = 0.75, lwd = 0.75) + 
        facet_wrap(~Organ, scales="free", nrow=3) + 
        labs(x = "", y = "Nucleolar fraction score", title = ctype) +
        scale_fill_manual(values = c("#00BFC4", "#F8766D", "#619CFF")) + 
        monocle3:::monocle_theme_opts() +
        coord_flip(ylim = ylim1) + 
        theme(title = element_text(color = "black", size = 10), 
              axis.text.x = element_text(color = "black", size = 12),
          legend.position = "none",
          axis.line = element_line(color = "black"), 
          axis.text.y = element_text(color = "black", size = 9.5),
          axis.ticks = element_line(color = "black"),
          axis.ticks.y = element_blank())




## Pancreatic acinar cells

In [ ]:
df_ctype_110_117 = df_ctype %>%
    filter(Development_day %in% c(110, 117)) %>%
    group_by(organ_ctype) %>%
    summarise(med_score_0.7_0.4 = median(med_score_0.7_0.4)) %>%
    arrange(-med_score_0.7_0.4)

In [ ]:
celltype_update_order = df_ctype_110_117 %>%
    pull(organ_ctype)

In [ ]:
which(celltype_update_order == 'Pancreas-Acinar cells')

In [ ]:
length(celltype_update_order)

In [ ]:
## Figure 7C

options(repr.plot.width=6, repr.plot.height=4, repr.plot.res=300)
df_ctype_110_117 %>%
    mutate(organ_ctype = factor(organ_ctype, levels = celltype_update_order)) %>%
    ggplot(., aes(x = organ_ctype, y = med_score_0.7_0.4)) + 
        geom_bar(stat = "identity") + 
        geom_bar(data = df_ctype_110_117 %>% filter(organ_ctype == 'Pancreas-Acinar cells'), stat = "identity",
                 fill = "red") + 
        scale_y_continuous(expand = c(0,0)) +
        labs(x = "", y = "") + 
        monocle3:::monocle_theme_opts() +
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 0), 
              legend.position = "none",
              axis.line = element_line(color = "black"), 
              axis.text.y = element_text(color = "black", size = 0),
              axis.ticks = element_line(color = "black"),
              axis.ticks.x = element_blank())

In [ ]:
quantile(df_all$score_pred_glmm_0.7_0.4, c(0.05, 0.1, 0.2, 0.5, 0.9, 0.95, 0.98))

In [ ]:
## Supplementary Figure 26

create_organ_plot <- function(organ_name, data) {
  
  df_sub <- data %>% filter(Organ == organ_name)
  
  min_val <- quantile(df_sub$score_pred_glmm_0.7_0.4, 0.1, na.rm = TRUE)
  max_val <- quantile(df_sub$score_pred_glmm_0.7_0.4, 0.9, na.rm = TRUE)
  mid_val <- (min_val + max_val) / 2
  
  ggplot(df_sub, aes(x = Main_cluster_umap_1, y = Main_cluster_umap_2)) +
    geom_point(aes(color = score_pred_glmm_0.7_0.4), 
               size = 0.4, 
               stroke = 0.2, 
               alpha = 0.7) +
    
    scale_colour_gradient2(
      low = muted("blue"), 
      mid = "grey90", 
      high = muted("red"), 
      midpoint = mid_val,
      limits = c(min_val, max_val), 
      oob = scales::squish,
      guide = guide_colorbar(title = "Nucleolar\nfraction", barwidth = 0.5, barheight = 5)
    ) +
    
    labs(title = organ_name) +

    theme(
        aspect.ratio = 1,
        panel.background = element_rect(fill = "white", color = NA),
        plot.background = element_rect(fill = "white", color = NA),
        axis.title = element_blank(),
        axis.text  = element_blank(),
        axis.ticks = element_blank(),
        axis.line  = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        legend.text = element_text(size = 10),
        plot.title = element_text(size = 14, hjust = 0.5)
    )
}

organs <- unique(df_all$Organ[!is.na(df_all$Organ)])
plot_list <- lapply(organs, create_organ_plot, data = df_all)


options(repr.plot.width = 12, repr.plot.height = 16, repr.plot.res = 300)
wrap_plots(plot_list, ncol = 3)

In [ ]:
multiple_celltypes = df_all %>%
    filter(!is.na(Organ_cell_lineage)) %>%
    mutate(celltype = str_split(Organ_cell_lineage, "-", simplify = T)[, 2]) %>%
    distinct(Organ_cell_lineage, .keep_all=T) %>%
    count(celltype) %>%
    arrange(-n) %>%
    filter(n > 5) %>% pull(celltype)

In [ ]:
df_ctype_organ = df_all %>%
    group_by(Organ, Organ_cell_lineage, celltype) %>%
    summarise(med_score_0.7_0.4 = median(score_pred_glmm_0.7_0.4), 
              n = n()) %>%
    na.omit() %>%
    filter(n >= 100) %>%
    as.data.frame() %>%
    arrange(-med_score_0.7_0.4)

In [ ]:
## Supplementary Figure 27

plot_list <- list()

for (ctype in multiple_celltypes) {
    
    df_sub = df_all %>%
        mutate(Organ_cell_lineage = factor(Organ_cell_lineage, 
                            levels = df_ctype_organ$Organ_cell_lineage)) %>%
        filter(!is.na(Organ_cell_lineage)) %>% 
        filter(grepl(ctype, Organ_cell_lineage))
    

    organ_ordered = df_sub %>%
        group_by(Organ) %>%
        summarise(med_score = median(score_pred_glmm_0.7_0.4, na.rm = TRUE)) %>%
        arrange(med_score) %>%
        pull(Organ)

    whisker_stats <- df_sub %>%
        group_by(Organ) %>%
        summarise(
            low = boxplot.stats(score_pred_glmm_0.7_0.4)$stats[1], 
            high = boxplot.stats(score_pred_glmm_0.7_0.4)$stats[5] 
        )
    
    global_low <- min(whisker_stats$low, na.rm = TRUE)
    global_high <- max(whisker_stats$high, na.rm = TRUE)
    
    ylim_final <- c(global_low - (0.05 * abs(global_low)), 
                    global_high + (0.05 * abs(global_high)))

    g = df_sub %>%
        mutate(Organ = factor(Organ, levels = organ_ordered)) %>%
        ggplot(., aes(x = Organ, y = score_pred_glmm_0.7_0.4)) + 
            geom_violin(aes(fill = Organ), alpha = 0.7, adjust = 2, width = 0.92) + 
            geom_boxplot(width = 0.3, outlier.shape = NA, fatten = 0.75, lwd = 0.75) + 
            labs(x = "", y = "Nucleolar fraction score", title = str_wrap(ctype, width = 20)) +
            monocle3:::monocle_theme_opts() +
            coord_flip(ylim = ylim_final) + 
            theme(legend.position = "none",
                  axis.title = element_text(color = "black", size = 11),
                  plot.title = element_text(color = "black", size = 11, hjust = 0.5), 
                  axis.text.x = element_text(color = "black", size = 9.5),
                  axis.line = element_line(color = "black"), 
                  axis.text.y = element_text(color = "black", size = 10.5),
                  axis.ticks = element_line(color = "black"),
                  plot.margin = margin(t = 10, r = 10, b = 25, l = 10))
    
    plot_list[[ctype]] <- g
}


options(repr.plot.width=7.5, repr.plot.height=11, repr.plot.res=300) 
wrap_plots(plot_list, ncol = 3) 

In [ ]:
df_upr <- df_all %>%
    filter(!is.na(Organ_cell_lineage) & upr_score > 0) %>%
    group_by(Organ, Organ_cell_lineage, Fetus_id, Development_day) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4),
              med_target = median(upr_score),
              n = n(), 
              .groups = "drop") %>%
    filter(n >= 100) %>%

    group_by(Organ, Organ_cell_lineage) %>%
    summarise(med_score = median(med_score),
              med_target = median(med_target),
              n = n(), 
              .groups = "drop") %>%
    mutate(score_type = "UPR")

df_granule <- df_all %>%
    filter(!is.na(Organ_cell_lineage) & granule_score > 0) %>%
    group_by(Organ, Organ_cell_lineage, Fetus_id, Development_day) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4),
              med_target = median(granule_score),
              n = n(), 
              .groups = "drop") %>%
    filter(n >= 100) %>%
    group_by(Organ, Organ_cell_lineage) %>%
    summarise(med_score = median(med_score),
              med_target = median(med_target),
              n = n(), 
              .groups = "drop") %>%
    mutate(score_type = "Stress granule")

df_combined <- bind_rows(df_upr, df_granule)

In [ ]:
## Figure 7G
options(repr.plot.width=3, repr.plot.height=4, repr.plot.res=300)

ggplot(df_combined, aes(x = med_score, y = med_target, color = score_type)) + 
    geom_point(size = 0.8, alpha = 0.5) + 
    geom_smooth(method = "lm", se = FALSE, linewidth = 1.5) +
    scale_color_manual(values = c("UPR" = "#c51b7d", "Stress granule" = "#4d9221")) +
    ylim(c(NA, 2.7)) + 
    labs(x = "", y = "", color = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "top", # Move legend to top so it doesn't squish the plot
          axis.text.x = element_text(color = "black", size = 14), 
          axis.text.y = element_text(color = "black", size = 14), 
          axis.title = element_text(color = "black", size = 14), 
          axis.ticks = element_line(color = "black"))

In [ ]:
correlation_stats <- df_combined %>%
  group_by(score_type) %>%
  summarise(
    pearson_r = cor(med_score, med_target, method = "pearson", use = "complete.obs"),
    p_value = cor.test(med_score, med_target, method = "pearson")$p.value,
    n_cells = n()
  )

print(correlation_stats)

In [ ]:
n_filter = 100
safe_cor_test <- possibly(cor.test, otherwise = NULL)


calculate_tissue_cor <- function(data, target_col, score_name) {
    target_sym <- sym(target_col)
    
    data %>%
        filter(!is.na(Organ_cell_lineage)) %>%
        filter(!!target_sym > 0) %>%
        group_by(Organ, Organ_cell_lineage, Fetus_id) %>%
        summarise(
            med_score = median(score_pred_glmm_0.7_0.4),
            med_target = median(!!target_sym),
            n = n(),
            .groups = "drop"
        ) %>%
        filter(n >= n_filter) %>%
        group_by(Organ, Organ_cell_lineage) %>%
        summarise(
            med_score = median(med_score),
            med_target = median(med_target),
            .groups = "drop"
        ) %>%
        group_by(Organ) %>%
        filter(n() >= 5) %>%
        tidyr::nest() %>%
        mutate(
            cor_model = map(data, ~ suppressWarnings(
                safe_cor_test(.x$med_score, .x$med_target, method = "spearman")
            ))
        ) %>%
        filter(!map_lgl(cor_model, is.null)) %>%
        mutate(
            score_type = score_name,
            spearman_rho = map_dbl(cor_model, "estimate"),
            p_value = map_dbl(cor_model, "p.value"),
            n_points = map_int(data, nrow)
        ) %>%
        select(score_type, Organ, spearman_rho, p_value, n_points)
}


upr_tissue_cor     <- calculate_tissue_cor(df_all, "upr_score", "UPR")
granule_tissue_cor <- calculate_tissue_cor(df_all, "granule_score", "Stress granule")

combined_tissue_cor <- bind_rows(upr_tissue_cor,granule_tissue_cor) %>%
    arrange(score_type, desc(spearman_rho))

combined_tissue_cor

In [ ]:
saveRDS(combined_tissue_cor, "../data/combined_tissue_cor_human_fetal_atlas.RDS")